# P0019 Experiment 02: LLM Response Collection

Collects outputs from Claude Sonnet and GPT-4o.

**Prerequisites:** Run exp01 first. Need API keys for Anthropic and OpenAI.

In [ ]:
# === Setup ===
from google.colab import drive, userdata
drive.mount('/content/drive')

!pip install -q anthropic openai

import json, os, time, csv
from datetime import datetime

BASE = '/content/drive/MyDrive/P0019'
OUT = f'{BASE}/exp02_collect'
os.makedirs(OUT, exist_ok=True)

with open(f'{BASE}/exp01_stimulus/exp01_pilot_manifest.json') as f:
    pilot_trials = json.load(f)
with open(f'{BASE}/exp01_stimulus/exp01_trial_manifest.json') as f:
    all_trials = json.load(f)
print(f"Pilot: {len(pilot_trials)} | Full: {len(all_trials)}")

In [ ]:
# === API Clients ===
import anthropic, openai

try:
    ANTHROPIC_KEY = userdata.get('ANTHROPIC_API_KEY')
except:
    ANTHROPIC_KEY = input("Anthropic API key: ")
try:
    OPENAI_KEY = userdata.get('OPENAI_API_KEY')
except:
    OPENAI_KEY = input("OpenAI API key: ")

claude_client = anthropic.Anthropic(api_key=ANTHROPIC_KEY)
openai_client = openai.OpenAI(api_key=OPENAI_KEY)
print("API clients ready.")

In [ ]:
# === Collection Engine ===

def call_claude(sys_p, usr_p):
    resp = claude_client.messages.create(
        model="claude-sonnet-4-20250514", max_tokens=4096, temperature=0,
        system=sys_p, messages=[{"role":"user","content":usr_p}])
    return resp.content[0].text, {"in": resp.usage.input_tokens, "out": resp.usage.output_tokens}

def call_gpt4o(sys_p, usr_p):
    resp = openai_client.chat.completions.create(
        model="gpt-4o", max_tokens=4096, temperature=0,
        messages=[{"role":"system","content":sys_p}, {"role":"user","content":usr_p}])
    return resp.choices[0].message.content, {"in": resp.usage.prompt_tokens, "out": resp.usage.completion_tokens}

def collect_responses(trials, output_path, log_path, resume=True):
    existing = {}
    if resume and os.path.exists(output_path):
        with open(output_path) as f:
            existing = {r["trial_id"]: r for r in json.load(f)}
        print(f"Resuming: {len(existing)} done.")

    results = list(existing.values())
    log_f = open(log_path, 'a', newline='')
    log_w = csv.writer(log_f)
    if not os.path.exists(log_path) or os.path.getsize(log_path) == 0:
        log_w.writerow(["trial_id","timestamp","model","status","in_tokens","out_tokens","sec","error"])

    pending = [t for t in trials if t["trial_id"] not in existing]
    print(f"To run: {len(pending)} / {len(trials)}")

    for i, trial in enumerate(pending):
        t0 = time.time()
        try:
            if "claude" in trial["model"]:
                text, usage = call_claude(trial["system_prompt"], trial["user_prompt"])
            else:
                text, usage = call_gpt4o(trial["system_prompt"], trial["user_prompt"])
            elapsed = time.time() - t0
            result = {**{k: trial[k] for k in ["trial_id","domain","v_level","v_magnitude",
                        "psi_direction","model","repetition","concept_order"]},
                      "response_text": text, "input_tokens": usage["in"],
                      "output_tokens": usage["out"], "elapsed_sec": round(elapsed,2),
                      "timestamp": datetime.now().isoformat(), "status": "success"}
            results.append(result)
            log_w.writerow([trial["trial_id"], datetime.now().isoformat(), trial["model"],
                           "success", usage["in"], usage["out"], round(elapsed,2), ""])
            if (i+1) % 5 == 0 or (i+1) == len(pending):
                with open(output_path, 'w') as f:
                    json.dump(results, f, indent=1, ensure_ascii=False)
            print(f"  [{i+1}/{len(pending)}] {trial['trial_id']}: {usage['out']} tokens, {elapsed:.1f}s")
            time.sleep(0.5)
        except Exception as e:
            log_w.writerow([trial["trial_id"], datetime.now().isoformat(), trial["model"],
                           "error", 0, 0, round(time.time()-t0,2), str(e)])
            print(f"  [{i+1}/{len(pending)}] ERROR: {e}")
            time.sleep(2)
    log_f.close()
    with open(output_path, 'w') as f:
        json.dump(results, f, indent=1, ensure_ascii=False)
    print(f"Done: {len(results)} results saved.")
    return results

## Run Pilot (~30 trials, ~$1-3)
Run this first to verify everything works.

In [ ]:
pilot_results = collect_responses(
    pilot_trials,
    f'{OUT}/exp02_pilot_responses.json',
    f'{OUT}/exp02_pilot_log.csv')

In [ ]:
# Quick check
from collections import Counter
for v in ["V0","V1","V2","V3"]:
    sub = [r for r in pilot_results if r["v_level"]==v]
    if sub:
        avg = sum(r["output_tokens"] for r in sub)/len(sub)
        print(f"{v}: {len(sub)} trials, avg {avg:.0f} tokens")
print("\nSample V0 (first 200 chars):")
print([r for r in pilot_results if r["v_level"]=="V0"][0]["response_text"][:200])
print("\nSample V3 (first 200 chars):")
print([r for r in pilot_results if r["v_level"]=="V3"][0]["response_text"][:200])

## Full Experiment (~400 trials, ~$10-25)
Uncomment after pilot verification.

In [ ]:
# === FULL RUN (uncomment to execute) ===
# full_results = collect_responses(
#     all_trials,
#     f'{OUT}/exp02_full_responses.json',
#     f'{OUT}/exp02_full_log.csv')
print("Uncomment above to run full experiment.")